# Digital Image Processing Lab - Assignment 3
**Problem Statements:**
1. **WAP to draw a border around the input image (or in other words pad the input image):**
   - Input the width of the border (in terms of pixel count) and the desired color of the border from the user.
   - Try this for both binary and grayscale image.
2. **WAP for Image Compliment (Negative):**
   - Try this for both binary and grayscale image.

---
### Theoretical Foundations
1. **Image Padding / Border Drawing:**
   Given an input image $I$ of dimensions $H \times W$, padding by width $P$ creates a new image $I_{\text{padded}}$ of dimensions $(H + 2P) \times (W + 2P)$ such that:
   $$I_{\text{padded}}(x, y) = \begin{cases} I(x - P, y - P) & \text{if } P \le x < H + P \text{ and } P \le y < W + P \\ C_{\text{border}} & \text{otherwise} \end{cases}$$

2. **Image Complement (Negative):**
   Reverses the intensity levels:
   - For Grayscale ($L=256$): $I_{\text{comp}}(x, y) = (L - 1) - I(x, y) = 255 - I(x, y)$
   - For Binary ($0$ or $255$): $B_{\text{comp}}(x, y) = 255 - B(x, y)$ (or bitwise NOT $\sim B$)


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Images Generator
def create_test_images():
    # Grayscale image
    x = np.linspace(0, 1, 200)
    y = np.linspace(0, 1, 200)
    xx, yy = np.meshgrid(x, y)
    gray = (np.sin(xx * 10) * np.cos(yy * 10) * 127 + 128).astype(np.uint8)
    cv2.putText(gray, "GRAY", (45, 115), cv2.FONT_HERSHEY_SIMPLEX, 1.3, 255, 3)

    # Binary image
    binary = np.zeros((200, 200), dtype=np.uint8)
    cv2.circle(binary, (100, 100), 65, 255, -1)
    cv2.rectangle(binary, (30, 30), (80, 80), 255, -1)
    cv2.putText(binary, "BIN", (70, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, 0, 2)
    
    return gray, binary

gray_img, bin_img = create_test_images()
print("Sample images prepared successfully.")


In [ ]:
# Cell 3: Task 1 - Border / Padding Implementation
# User Inputs
pad_width = 25 #@param {type:"integer"}
gray_border_color = 180 #@param {type:"integer"}  (0=black, 255=white, 128=gray)
bin_border_color = 255 #@param [0, 255] {type:"raw"}

def add_border_manual(image, pad_w, color):
    """Pads image from scratch by allocating an expanded canvas."""
    h, w = image.shape
    new_h = h + 2 * pad_w
    new_w = w + 2 * pad_w
    padded = np.full((new_h, new_w), color, dtype=image.dtype)
    padded[pad_w:pad_w+h, pad_w:pad_w+w] = image
    return padded

# Apply manual padding
padded_gray_scratch = add_border_manual(gray_img, pad_width, gray_border_color)
padded_bin_scratch = add_border_manual(bin_img, pad_width, bin_border_color)

# Verification with OpenCV cv2.copyMakeBorder
padded_gray_cv = cv2.copyMakeBorder(gray_img, pad_width, pad_width, pad_width, pad_width, 
                                     cv2.BORDER_CONSTANT, value=gray_border_color)
print(f"Scratch padding matches OpenCV: {np.array_equal(padded_gray_scratch, padded_gray_cv)}")


In [ ]:
# Cell 4: Task 2 - Image Complement Implementation
def compute_complement_manual(image):
    """Computes image negative/complement from scratch."""
    return (255 - image).astype(np.uint8)

# Compute complements
gray_comp_scratch = compute_complement_manual(gray_img)
bin_comp_scratch = compute_complement_manual(bin_img)

# Verify with bitwise NOT
gray_comp_cv = cv2.bitwise_not(gray_img)
bin_comp_cv = cv2.bitwise_not(bin_img)

print(f"Grayscale complement matches OpenCV: {np.array_equal(gray_comp_scratch, gray_comp_cv)}")
print(f"Binary complement matches OpenCV: {np.array_equal(bin_comp_scratch, bin_comp_cv)}")


In [ ]:
# Cell 5: Comprehensive Visualizations
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Row 1: Grayscale operations
axes[0, 0].imshow(gray_img)
axes[0, 0].set_title("Original Grayscale", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(padded_gray_scratch)
axes[0, 1].set_title(f"Padded Grayscale\n(width={pad_width}, color={gray_border_color})", fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(gray_comp_scratch)
axes[0, 2].set_title("Grayscale Complement (255 - I)", fontweight='bold')
axes[0, 2].axis('off')

axes[0, 3].hist(gray_img.ravel(), bins=64, alpha=0.6, label='Original', color='blue')
axes[0, 3].hist(gray_comp_scratch.ravel(), bins=64, alpha=0.6, label='Complement', color='red')
axes[0, 3].set_title("Grayscale vs Complement Hist", fontweight='bold')
axes[0, 3].legend()
axes[0, 3].grid(True, alpha=0.3)

# Row 2: Binary operations
axes[1, 0].imshow(bin_img)
axes[1, 0].set_title("Original Binary", fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(padded_bin_scratch)
axes[1, 1].set_title(f"Padded Binary\n(width={pad_width}, color={bin_border_color})", fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(bin_comp_scratch)
axes[1, 2].set_title("Binary Complement (~B)", fontweight='bold')
axes[1, 2].axis('off')

axes[1, 3].axis('off')
text_summary = (
    "Task Summary:\n\n"
    f"• Padding Width: {pad_width} px\n"
    f"• Gray Border Color: {gray_border_color}\n"
    f"• Binary Border Color: {bin_border_color}\n\n"
    "• Verification:\n"
    "  - Padding verified against cv2.copyMakeBorder\n"
    "  - Complement verified against cv2.bitwise_not"
)
axes[1, 3].text(0.1, 0.4, text_summary, fontsize=12, family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()


### Conclusions
- Image padding successfully expands the spatial dimensions without modifying original interior content, essential for spatial convolution and boundary preservation.
- Image complement inverts tonal dynamics, widely used for enhancing medical X-rays and photographic negatives.
